In [1]:
from numpy.typing.mypy_plugin import plugin

import model
import plotter
import sys
import json
import math
import warnings
import pandas as pd
import plotly
import sqlite3
import plotly.express as px
import plotly.graph_objects as go
from pyathena import connect
import datetime
import os
import glob

from plotter import FileRunsInfo, DBRunsInfo, RunInfo

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', None)

from IPython.display import display, HTML
display(HTML("<style>.jp-Cell { margin-left: -20% !important; margin-right: -15% !important; }</style>"))

sql_con = sqlite3.connect("data.db")

import matplotlib.pyplot as plt

In [2]:
ops = ['All Reduce', 'All Gather', 'Reduce Scatter', 'All to All']
file_name_mapping = {
    'All Reduce':'all_reduce', 
    'All Gather':'all_gather', 
    'Reduce Scatter':'reduce_scatter',
    'All to All':'alltoall'
}
masks = ['0x0', '0x7']

# ops = ['all_gather_perf','reduce_scatter_perf']
# masks = ['0x0']

plot = plotter.NCCLPlotter(sql_con)

def init_records():
    records = {}
    records['All Reduce'] = {'0x0': {}, '0x7': {}}
    records['All Gather'] = {'0x0': {}, '0x7': {}}
    records['Reduce Scatter'] = {'0x0': {}, '0x7': {}}
    records['All to All'] = {'0x0': {}, '0x7': {}}
    return records


def get_standard_runs_from_db(records, branch, key=True,
                      version=None,
                      instance='p5en.48xlarge',
                      plugin_version=None,
                      efa_installer=None,
                      os=None,
                      optimized=True, start_time=None, end_time=None):
    for op in ops:
        for mask in masks:
            records[op][mask][key] = plot.search_runs(
                branch, op, num_nodes=16, instance_type=instance, nccl_version=version, mask=mask,
                optimized=optimized, plugin_version=plugin_version, os=os, efa_version=efa_installer, start_time=start_time, end_time=end_time
            )
            
            
un = 'unknown'
def get_standard_runs_from_files(records, paths, key=True, instance=un, plugin_version=un, efa_installer=un, nccl_version=un, timestamp=datetime.datetime.now().timestamp()):
    slurmouts = []
    for path in paths:
        slurmouts += glob.glob(f'{path}/**/slurmout*')

    slurmouts.sort(key=os.path.getmtime)
    
    for op in ops:
        for mask in masks:
            found = []
            match = f'{file_name_mapping[op]}_perf-16-{mask}'
            for file in slurmouts:
                if match in file: found.append(file)
            records[op][mask][key] = FileRunsInfo(op, mask, found, instance=instance, nccl_version=nccl_version, plugin_version=plugin_version, efa_installer=efa_installer,timestamp=timestamp)

def graph_standard_runs(records, keys=[]):
    for op in ops:
        for mask in masks:
            record = records[op][mask] 
            bw_options = plotter.PlotOptions(bw=True, title=f'{op} {mask} Bandwidth')
            fig = plot.make_plot(bw_options)
            bw_options.filter_func = plotter.DataSizeFilter.last_n
            for key in keys:
                bw_options.annotation = f'{key}'
                runs = record[key]
                if type(runs) is FileRunsInfo: t = plot.add_plots_from_files(runs, bw_options)
                elif type(runs) is DBRunsInfo: t = plot.add_plots_from_db(runs, bw_options)  
            if op in ('All Reduce', 'All Gather', 'Reduce Scatter') and mask in ('0x0', '0x7'):
                print('Modeling', op)
                info = RunInfo(
                    op, mask,
                    nodes = 16,
                    timestamp = datetime.datetime.now().timestamp(), 
                    instance = 'p5en.48xlarge', 
                    os = 'Model', 
                    plugin_version = 'Model', 
                    nccl_version = 'Model', 
                    efa_installer = 'Model', 
                    aws_ofi_nccl_commit = 'Model'
                )
                bw_options.annotation = f'Roofline Model (p5en)'
                bw_options.instance = 'p5en.48xlarge'
                t = plot.add_plot_from_model(bw_options, info)
            plot.show_plot()
            plot.save_plot(f'{op} {mask} Bandwidth.png')

            lat_options = plotter.PlotOptions(bw=False, title=f'{op} {mask} Latency')
            fig = plot.make_plot(lat_options)
            lat_options.filter_func = plotter.DataSizeFilter.first_n
            for key in keys:
                lat_options.annotation = f'{key}'
                runs = record[key]
                if type(runs) is FileRunsInfo: t = plot.add_plots_from_files(runs, lat_options)
                elif type(runs) is DBRunsInfo: t = plot.add_plots_from_db(runs, lat_options)
            if op in ('All Reduce', 'All Gather', 'Reduce Scatter') and mask in ('0x0', '0x7'):
                info = RunInfo(
                    op, mask,
                    nodes = 16,
                    timestamp = datetime.datetime.now().timestamp(), 
                    instance = 'p5en.48xlarge', 
                    os = 'Model', 
                    plugin_version = 'Model', 
                    nccl_version = 'Model', 
                    efa_installer = 'Model', 
                    aws_ofi_nccl_commit = 'Model'
                )
                lat_options.annotation = f'Roofline Model (p5en)'
                lat_options.instance = 'p5en.48xlarge'
                t = plot.add_plot_from_model(lat_options, info)
            plot.show_plot()
            plot.save_plot(f'{op} {mask} Latency.png') 






In [3]:
records = init_records()

get_standard_runs_from_db(records, 'release_branch', key='p5en, 2.27.6, 1.43', version='v2.27%',
                          instance='p5en.48xlarge', optimized=False, plugin_version="v1.16.x", start_time='2025-07-07')

graph_standard_runs(records, ['p5en, 2.27.6, 1.43'])

Modeling All Reduce


Modeling All Reduce


Modeling All Gather


Modeling All Gather


Modeling Reduce Scatter


Modeling Reduce Scatter
